In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [55]:
from pathlib import Path
import time

In [56]:
DIR = Path("basketball_data")
DIR.mkdir(exist_ok=True)

SEASONS = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26",
]

In [57]:
from nba_api.stats.endpoints import (
    leaguedashplayerstats,
    leaguedashlineups,
)

In [58]:
def fetch_player_stats(season, measure_type="Base"):
    endpoints = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star="Regular Season",
        measure_type_detailed_defense=measure_type,
        per_mode_detailed="PerGame",
        timeout=60,
    )

    return endpoints.get_data_frames()[0]

player_base = fetch_player_stats(SEASONS, "Base")

print("Rows:", len(player_base))
print("Columns:", len(player_base.columns))

player_base.head()

Rows: 605
Columns: 69


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,FP_HIGH_SCORE_RANK,TEAM_COUNT
0,203932,Aaron Gordon,Aaron,1610612743,DEN,26.0,75,46,29,0.613,...,203,87,91,36,103,85,40,105,104,1
1,1630565,Aaron Henry,Aaron,1610612755,PHI,22.0,6,6,0,1.000,...,560,564,578,506,591,268,40,591,589,1
2,1628988,Aaron Holiday,Aaron,1610612756,PHX,25.0,63,34,29,0.540,...,344,298,322,312,344,268,40,344,320,2
3,1630174,Aaron Nesmith,Aaron,1610612738,BOS,22.0,52,32,20,0.615,...,376,462,442,234,478,268,40,470,478,1
4,1630598,Aaron Wiggins,Aaron,1610612760,OKC,23.0,50,13,37,0.260,...,240,269,240,543,286,268,40,278,285,1


In [59]:
player_advanced = fetch_player_stats(SEASONS, "Advanced")
player_advanced.head()

,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,E_PACE_RANK,PACE_RANK,sp_work_PACE_RANK,PIE_RANK,FGM_RANK,FGA_RANK,FGM_PG_RANK,FGA_PG_RANK,FG_PCT_RANK,TEAM_COUNT
0,203932,Aaron Gordon,Aaron,1610612743,DEN,26.0,75,46,29,0.613,...,354,463,463,218,55,69,81,106,106,1
1,1630565,Aaron Henry,Aaron,1610612755,PHI,22.0,6,6,0,1.000,...,26,47,47,596,561,557,573,581,569,1
2,1628988,Aaron Holiday,Aaron,1610612756,PHX,25.0,63,34,29,0.540,...,428,400,400,272,257,250,320,310,287,2
3,1630174,Aaron Nesmith,Aaron,1610612738,BOS,22.0,52,32,20,0.615,...,318,312,312,510,361,351,440,434,443,1
4,1630598,Aaron Wiggins,Aaron,1610612760,OKC,23.0,50,13,37,0.260,...,239,208,208,461,254,251,236,240,223,1


In [60]:
all_player_stats = []
all_lineups = []

for SEASON in SEASONS:
    print("Gathering", SEASON)

    player_base = fetch_player_stats(SEASON, "Base")
    player_advanced = fetch_player_stats(SEASON, "Advanced")

    advanced_extra_cols = [
        col for col in player_advanced.columns
        if col not in player_base.columns
        and col != "PLAYER_ID"
    ]

    player_stats = player_base.merge(
        player_advanced[["PLAYER_ID"] + advanced_extra_cols],
        on="PLAYER_ID",
        how="left",
        validate="one_to_one",
    )

    player_stats["SEASON"] = SEASON

    player_stats.to_csv(
        DIR / f"player_stats_{SEASON}.csv",
        index=False,
    )

    all_player_stats.append(player_stats)

    lineups = fetch_lineup_stats(SEASON)
    lineups["SEASON"] = SEASON

    lineups.to_csv(
        DIR / f"lineups_{SEASON}.csv",
        index=False,
    )

    all_lineups.append(lineups)

    time.sleep(2)

Gathering 2021-22
Gathering 2022-23
Gathering 2023-24
Gathering 2024-25
Gathering 2025-26


In [61]:
players = pd.concat(all_player_stats, ignore_index=True)

players.to_csv(
    DIR / "player_stats_five_seasons.csv",
    index=False,
)

print("Saved player statistics.")

Saved player statistics.


In [62]:

print("Duplicate player-season IDs:", players.duplicated(["PLAYER_ID", "SEASON"]).sum())

print("\nMissing values in most incomplete columns:")
display(
    players.isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)


Duplicate player-season IDs: 0

Missing values in most incomplete columns:


PLAYER_ID            0
PLAYER_NAME          0
NICKNAME             0
TEAM_ID              0
TEAM_ABBREVIATION    0
AGE                  0
GP                   0
W                    0
L                    0
W_PCT                0
MIN                  0
FGM                  0
FGA                  0
FG_PCT               0
FG3M                 0
dtype: int64

In [63]:
player_base_per_game = fetch_player_stats(
    SEASONS[-1],
    "Base"
)

In [64]:
print("Players with the most minutes per game in the latest season:")

display(
    player_base_per_game.sort_values("MIN", ascending=False)[
        ["PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "MIN", "PTS", "AST"]
    ].head(10)
)

Players with the most minutes per game in the latest season:


,PLAYER_NAME,TEAM_ABBREVIATION,GP,MIN,PTS,AST
554,Tyrese Maxey,PHI,70,38.0,28.3,6.6
21,Amen Thompson,HOU,79,37.4,18.3,5.3
333,Kevin Durant,HOU,78,36.4,26.0,4.8
526,Toby Okani,MEM,6,36.2,10.0,1.0
372,Luka Dončić,LAL,64,35.8,33.5,8.3
542,Trey Murphy III,NOP,66,35.5,21.5,3.8
242,Jamal Murray,DEN,75,35.4,25.4,7.1
234,Jalen Johnson,ATL,72,35.2,22.5,7.9
218,Jabari Smith Jr.,HOU,77,35.1,15.8,1.9
31,Anthony Edwards,MIN,61,35.0,28.8,3.7


In [65]:
def fetch_lineup_stats(season, measure_type="Advanced"):
    endpoints = leaguedashlineups.LeagueDashLineups(
        season=season,
        season_type_all_star="Regular Season",
        measure_type_detailed_defense=measure_type,
        per_mode_detailed="Per100Possessions",
        group_quantity=5,
        timeout=60,
    )

    return endpoints.get_data_frames()[0]

lineups = fetch_lineup_stats(SEASON)

print("Rows:", len(lineups))
print("Columns:", len(lineups.columns))

lineups.head()

Rows: 2000
Columns: 50


,GROUP_SET,GROUP_ID,GROUP_NAME,TEAM_ID,TEAM_ABBREVIATION,GP,W,L,W_PCT,MIN,...,AST_RATIO_RANK,OREB_PCT_RANK,DREB_PCT_RANK,REB_PCT_RANK,TM_TOV_PCT_RANK,EFG_PCT_RANK,TS_PCT_RANK,PACE_RANK,PIE_RANK,SUM_TIME_PLAYED
0,Lineups,-203497-203944-1628978-1630162-1630183-,R. Gobert - J. Randle - D. DiVincenzo - A. Edw...,1610612750,MIN,54,33,21,0.611,710.0,...,7586,7743,8283,7230,12224,6964,7423,12412,8142,2129110
1,Lineups,-1626157-1628384-1628404-1628969-1628973-,K. Towns - O. Anunoby - J. Hart - M. Bridges -...,1610612752,NYK,43,30,13,0.698,541.0,...,5320,9262,7136,7422,11547,7209,7622,12988,8378,1621680
2,Lineups,-202699-1629130-1630595-1631105-1641709-,T. Harris - D. Robinson - C. Cunningham - J. D...,1610612765,DET,42,29,13,0.690,536.0,...,5748,5963,8176,6797,11260,7608,8288,12331,7912,1609130
3,Lineups,-1628970-1630163-1631217-1641706-1642851-,M. Bridges - L. Ball - M. Diabaté - B. Miller ...,1610612766,CHA,49,34,15,0.694,509.0,...,5253,5920,8071,6505,11259,5687,6155,13741,6719,1527740
4,Lineups,-203468-1629638-1630168-1630552-1630700-,C. McCollum - N. Alexander-Walker - O. Okongwu...,1610612737,ATL,30,20,10,0.667,391.0,...,5653,7634,7134,7238,10540,7001,7439,11022,7338,1171820


In [66]:
lineups = pd.concat(all_lineups, ignore_index=True)

lineups.to_csv(
    DIR / "lineups_five_seasons.csv",
    index=False,
)

print("Saved lineup statistics.")

Saved lineup statistics.


In [67]:
display(
    lineups[lineups["SEASON"] == SEASONS[-1]]
    .sort_values("MIN", ascending=False)[
        [c for c in [
            "GROUP_NAME",
            "TEAM_ABBREVIATION",
            "GP",
            "MIN",
            "OFF_RATING",
            "DEF_RATING",
            "NET_RATING",
        ] if c in lineups.columns]
    ]
    .head(20)
)

,GROUP_NAME,TEAM_ABBREVIATION,GP,MIN,OFF_RATING,DEF_RATING,NET_RATING
8000,R. Gobert - J. Randle - D. DiVincenzo - A. Edw...,MIN,54,710.0,118.8,110.9,7.9
8001,K. Towns - O. Anunoby - J. Hart - M. Bridges -...,NYK,43,541.0,115.0,112.8,2.3
8002,T. Harris - D. Robinson - C. Cunningham - J. D...,DET,42,536.0,121.4,109.9,11.4
8003,M. Bridges - L. Ball - M. Diabaté - B. Miller ...,CHA,49,509.0,134.9,108.6,26.4
8004,C. McCollum - N. Alexander-Walker - O. Okongwu...,ATL,30,391.0,123.1,102.8,20.3
8005,K. Durant - J. Okogie - A. Sengun - J. Smith J...,HOU,41,377.0,121.8,111.5,10.3
8006,A. Gordon - N. Jokić - J. Murray - C. Johnson ...,DEN,23,360.0,127.0,113.4,13.5
8007,B. Ingram - J. Poeltl - R. Barrett - I. Quickl...,TOR,28,354.0,122.6,115.0,7.6
8008,M. Porter Jr. - T. Mann - N. Claxton - N. Clow...,BKN,19,307.0,116.7,114.5,2.2
8009,K. Durant - A. Sengun - J. Smith Jr. - T. Easo...,HOU,36,306.0,116.4,107.3,9.1
